# Competitive Intelligence Report Generator | Orchestrator-Worker

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from typing import TypedDict, List, Annotated
import operator
import json
import re
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
class OrchestratorState(TypedDict):
    company: str
    subtasks: List[str]
    worker_results: Annotated[List[str], operator.add]  # Accumulate results from parallel Send()
    final_report: str

class WorkerInput(TypedDict):
    company: str
    subtask: str

In [5]:
def parse_json(text: str):
    """Extract and parse JSON from LLM output, handling markdown fences."""
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", text).strip()
    return json.loads(cleaned)

In [6]:
# Orchestrator: dynamically plan research subtasks based on the company
def plan_subtasks(state: OrchestratorState) -> dict:
    response = model.invoke(
        f"You are a competitive intelligence analyst. Given a company, identify 3-4 key research areas "
        f"needed for a thorough competitive analysis. Each subtask should be specific and actionable.\n\n"
        f"Company: {state['company']}\n\n"
        f"Return a JSON list of strings. Example:\n"
        f'["Analyze revenue streams and financial performance", "Evaluate product portfolio", ...]'
    )
    parsed = parse_json(response.content)
    if not isinstance(parsed, list):
        parsed = [str(parsed)]
    subtasks = [str(item) for item in parsed]
    return {"subtasks": subtasks}

In [7]:
# Fan-out: use LangGraph's Send to dynamically spawn one worker per subtask
def route_to_workers(state: OrchestratorState) -> list[Send]:
    """LangGraph-native dynamic fan-out — spawns parallel worker nodes via Send()."""
    return [
        Send("execute_task", {"company": state["company"], "subtask": t})
        for t in state["subtasks"]
    ]

# Worker: each Send() invocation runs this node independently and in parallel
def execute_task(state: WorkerInput) -> dict:
    response = model.invoke(
        f"You are a business research analyst. Complete this research task thoroughly. "
        f"Provide specific data points, market context, and insights.\n\n"
        f"Company being analyzed: {state['company']}\n"
        f"Research task: {state['subtask']}"
    )
    return {"worker_results": [response.content]}

In [8]:
# Synthesizer: combine all research into a structured report
def synthesize_report(state: OrchestratorState) -> dict:
    all_research = "\n\n---\n\n".join(
        f"## {task}\n\n{result}"
        for task, result in zip(state["subtasks"], state["worker_results"])
    )
    response = model.invoke(
        f"You are a senior business analyst. Synthesize the following research into a professional "
        f"Competitive Intelligence Report for {state['company']}.\n\n"
        f"Structure the report as:\n"
        f"1. Executive Summary\n"
        f"2. Key Findings (from each research area)\n"
        f"3. Competitive Advantages & Weaknesses\n"
        f"4. Strategic Recommendations\n\n"
        f"Research:\n{all_research}"
    )
    return {"final_report": response.content}

In [9]:
# Build graph: plan -> fan-out via Send -> execute_task (parallel) -> synthesize
graph = StateGraph(OrchestratorState)
graph.add_node("plan", plan_subtasks)
graph.add_node("execute_task", execute_task)
graph.add_node("synthesize", synthesize_report)
graph.add_edge(START, "plan")
graph.add_conditional_edges("plan", route_to_workers)  # Dynamic fan-out
graph.add_edge("execute_task", "synthesize")
graph.add_edge("synthesize", END)

orchestrator = graph.compile()

In [10]:
# Plot the workflow
plot_mermaid(orchestrator)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__(<p>__start__</p>)
	plan(plan)
	execute_task(execute_task)
	synthesize(synthesize)
	__end__(<p>__end__</p>)
	__start__ --> plan;
	plan --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [11]:
result = orchestrator.invoke({"company": "Spotify"})
print(result["final_report"])

# Competitive Intelligence Report for Spotify

## 1. Executive Summary

Spotify stands as a global leader in the music streaming industry, driven by its dual revenue model—subscription and ad-supported services. The company differentiates itself through personalized user experiences and exclusive content offerings, including an expansive podcast library. Despite consistent revenue growth, Spotify faces profitability challenges due to high content costs, intensified market competition, and the necessity to enhance ad revenue. Strategic investments in podcasting and global expansion remain pivotal. This report outlines Spotify's competitive landscape, evaluating its advantages and weaknesses, and provides strategic recommendations for sustaining its market dominance.

## 2. Key Findings

### Revenue Diversification and Financial Performance
- Subscriptions (Spotify Premium) account for 88% of revenue, while ad-supported services contribute 12%.
- Spotify reported €6.2 billion in revenue 

In [12]:
stream_invoke(orchestrator, {"company": "Spotify"})

────────────────────────────────────────────────────────────────────────────────

  STREAMING EXECUTION

────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────

  EXECUTION COMPLETE

────────────────────────────────────────────────────────────────────────────────

{'company': 'Spotify',
 'subtasks': ['Analyze revenue diversification and financial performance, focusing on subscription vs. ad-supported models.',
  "Evaluate Spotify's market positioning and user growth trends in comparison to major competitors like Apple Music and Amazon Music.",
  "Investigate Spotify's investment and expansion strategies in podcasting and other audio content, including strategic partnerships and acquisitions.",
  'Assess consumer engagement and retention strategies, such as personalized playlist algorithms and exclusive content offerings.'],
 'worker_results': ["### Overview of Spotify's Revenue Streams\n\nSpotify, a leading player in the music streaming industry, derives its revenue through two primary channels: the subscription-based model (Spotify Premium) and the ad-supported model (Spotify Free). The success of these models is pivotal for Spotify's financial health and strategic business decisions.\n\n### Revenue Breakdown\n\n1. **Subscription-Based Model (S